# Robot-arm training: Colab platform preflight

Select **Runtime → Change runtime type → GPU**, then run these cells interactively.
This notebook uses the runtime's existing PyTorch. It does not install software,
mount Google Drive, fetch model weights, upload datasets, or start a long run.
It performs only 20 tiny synthetic optimizer steps and a checkpoint round-trip.

**This is a compute smoke test, not a task-policy training pilot.** It does not
qualify MolmoAct2, LeRobot, HIL-SERL, MuJoCo/EGL, private assets or real hardware.
The existing batch-8 MolmoAct2 recipe was measured near 30 GB; inspect the actual
GPU allocation before choosing a recipe. High system RAM is not GPU VRAM.

Colab GPUs, quotas and session lengths vary, including on paid plans. Save real
checkpoints to selected persistent storage before a session ends. Read the
[Colab FAQ](https://research.google.com/colaboratory/faq.html); free-tier
restrictions include remote/distributed workers and “chess training.” Our work
is robot manipulation, not chess-engine training, but free-tier suitability
should not be assumed. Do not create SSH tunnels or keep-alive workarounds.


In [ ]:
import json, platform, shutil, subprocess, sys, time
from pathlib import Path
import torch

if not torch.cuda.is_available():
    raise RuntimeError("No CUDA GPU. Select a GPU runtime; do not silently use CPU.")
report = {
    "schema": 1,
    "kind": "synthetic-compute-smoke-only",
    "python": sys.version,
    "torch": torch.__version__,
    "cuda_runtime": torch.version.cuda,
    "platform": platform.platform(),
    "gpu": torch.cuda.get_device_name(0),
    "capability": list(torch.cuda.get_device_capability(0)),
}
free, total = torch.cuda.mem_get_info()
report.update(vram_total_gib=total / 2**30, vram_free_gib=free / 2**30,
              disk_free_gib=shutil.disk_usage("/content" if Path("/content").exists() else ".").free / 2**30)
report["nvidia_smi"] = subprocess.check_output(
    ["nvidia-smi", "--query-gpu=name,driver_version,memory.total,memory.free", "--format=csv,noheader"],
    text=True).strip()
report["kernel_log_status"] = "not inspected; managed runtime visibility unknown"
print(json.dumps(report, indent=2))


In [ ]:
# A bounded, deterministic small-model optimizer/checkpoint smoke test.
# No simulator data, images, pretrained weights or credentials are used.
torch.manual_seed(20261030)
device = "cuda:0"
model = torch.nn.Sequential(torch.nn.Linear(6, 64), torch.nn.Tanh(), torch.nn.Linear(64, 6)).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
x = torch.randn(32, 6, device=device)
y = 0.5 * x
losses = []
torch.cuda.reset_peak_memory_stats()
torch.cuda.synchronize()
started = time.monotonic()
for step in range(20):
    optimizer.zero_grad(set_to_none=True)
    loss = torch.nn.functional.mse_loss(model(x), y)
    if not torch.isfinite(loss):
        raise RuntimeError(f"nonfinite loss at step {step}")
    loss.backward()
    if not all(p.grad is None or torch.isfinite(p.grad).all() for p in model.parameters()):
        raise RuntimeError(f"nonfinite gradient at step {step}")
    optimizer.step()
    losses.append(loss.item())
torch.cuda.synchronize()
report.update(optimizer_steps=20, elapsed_seconds=time.monotonic() - started,
              peak_allocated_gib=torch.cuda.max_memory_allocated() / 2**30,
              initial_loss=losses[0], final_loss=losses[-1])
checkpoint_path = Path("/content" if Path("/content").exists() else ".") / "robot-arm-preflight-checkpoint.pt"
expected = model(x).detach()
torch.save({"model":model.state_dict(), "optimizer":optimizer.state_dict(), "step":20}, checkpoint_path)
checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=True)
restored = torch.nn.Sequential(torch.nn.Linear(6, 64), torch.nn.Tanh(), torch.nn.Linear(64, 6)).to(device)
restored.load_state_dict(checkpoint["model"])
restored_optimizer = torch.optim.AdamW(restored.parameters(), lr=1e-3)
restored_optimizer.load_state_dict(checkpoint["optimizer"])
if checkpoint["step"] != 20 or not torch.equal(expected, restored(x).detach()):
    raise RuntimeError("checkpoint reload changed predictions or step count")
# Also take one resumed optimizer step to exercise restored optimizer state.
restored_optimizer.zero_grad(set_to_none=True)
resume_loss = torch.nn.functional.mse_loss(restored(x), y)
resume_loss.backward()
if not torch.isfinite(resume_loss) or not all(p.grad is None or torch.isfinite(p.grad).all() for p in restored.parameters()):
    raise RuntimeError("nonfinite resumed optimizer step")
restored_optimizer.step()
if not all(torch.isfinite(p).all() for p in restored.parameters()):
    raise RuntimeError("nonfinite resumed parameters")
report.update(checkpoint_reload_exact=True, resumed_optimizer_step=True,
              checkpoint_bytes=checkpoint_path.stat().st_size,
              passed=True, scope="CUDA/synthetic optimizer only; task training NOT qualified")
report_path = checkpoint_path.with_name("robot-arm-colab-preflight.json")
report_path.write_text(json.dumps(report, indent=2) + "\n")
print(json.dumps(report, indent=2))
print(f"Saved {report_path}; share this report to continue platform qualification.")


## Next gate (not automatically executed)

Return the JSON report to the robot-arm project session. Choose the **task/model**
and persistent private storage from the measured hardware. Then create a separate
pinned LeRobot environment, export only the audited training split, and run the
20-step **actual-task** training/checkpoint pilot before budgeting a long run.
No cloud transfer of private recordings/CAD is implied by this smoke test.

For HIL-SERL, the [official simulation guide](https://huggingface.co/docs/lerobot/main/en/hilserl_sim)
uses `gym_hil`/Franka Panda. Our SO101 scene requires a compatible environment
adapter, observations/actions, reward/termination/reset contract, interventions
and evaluation. `main` documentation is not a compatibility guarantee for our
historically pinned LeRobot commit. Keep HIL-SERL in a separately pinned runtime.
